In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Abstract
This study found that utility prices in Singapore were about 6 times more expensive than that of the US

# Initial Hypothesis:
I suspect that the cost of utilities in the US and SG would simply differ by a correction factor.

# Goal
Market data on three different utilities: Water, Electricity and Natural Gas, were obtained from the US and SG. 
 We fit 2 different Simple Linear Regression Models (SLR) to the data for each utility pair (US and SG), an SLR with an intercept and another SLR without. It is important to note that a rebasing was done, all calculations were done in USD (SGD values were converted to USD)

# Observations and Results
- Across all the utility models, it is observed that all the fitted Simple Linear Regression models without intercept had better performance that the models with intercept. This matches our initial intuition that the utilities of the US and the SG would differ by a scaling factor.
- The correction factor calculated to be applied to our economic analysis (to scale the price of our utilities to be more applicable to a context in SG) was simply taken to be the average of the correction factors for the three utilities. THe correction factor taken in this case is simply the inverse of the coefficient of the predictor. (Calculations shown below)
    - For Electricity
      - Coefficient of SG_Price = 0.2268
    - For Natural Gas
      - Coefficient of Price_SG = 0.2379
    - For Water 
      - Coefficient of SG_Price_per_m3 = 0.0148
Taking the average of the three we get: 0.15983. And by taking the inverse, we can obtain the correction factor (to be multipled to our US utility prices). The correction factor would be ~6.256. This means that on average, utility prices in Singapore are about 6.256 times more expensive than that of the US. 

In [102]:
# Reading in data here

US_ng_data = pd.read_csv("data/us_monthly_NG_prices.csv", parse_dates=['Date'])

SG_ng_data = pd.read_csv("data/asia_monthly_NG_prices.csv", parse_dates=['observation_date'])
SG_ng_data.rename(columns={'PNGASJPUSDM': 'Price'}, inplace=True)


## CALCULATING SCALING FACTOR FOR ELECTRICITY

In [115]:
# Writing Directly from google drive

US_electricity_prices = pd.DataFrame([[2016,6.76],[2017,6.88],[2018,6.92],
                         [2019,6.81],[2020,6.67],[2021,7.18],
                         [2022,8.32],[2023,8.04],[2024,8.13],[2025,8.62]], columns=['Year', 'US_Price'])

SG_electricity_prices = pd.DataFrame([[2019,30.208],[2020,28.288],[2021,29.184],
                         [2022,36.352],[2023,36.096],[2024,38.144]], columns=['Year', 'SG_Price'])

# print(SG_electricity_prices.head())
# print(US_electricity_prices.head())

   Year  SG_Price
0  2019    30.208
1  2020    28.288
2  2021    29.184
3  2022    36.352
4  2023    36.096
   Year  US_Price
0  2016      6.76
1  2017      6.88
2  2018      6.92
3  2019      6.81
4  2020      6.67


In [117]:
combined_df = pd.merge(US_electricity_prices, SG_electricity_prices, on='Year', how='outer').dropna(inplace=False)
print(combined_df)

   Year  US_Price  SG_Price
3  2019      6.81    30.208
4  2020      6.67    28.288
5  2021      7.18    29.184
6  2022      8.32    36.352
7  2023      8.04    36.096
8  2024      8.13    38.144


In [118]:
# Regression with intercept (includes a constant term)
X_with_intercept = sm.add_constant(combined_df['SG_Price'])  # Predictor with intercept
model_with_intercept = sm.OLS(combined_df['US_Price'], X_with_intercept).fit()  # Response: Price_US
print("Regression Model WITH Intercept:")
print(model_with_intercept.summary())

# Regression without intercept (no constant term)
model_without_intercept = sm.OLS(combined_df['US_Price'], combined_df['SG_Price']).fit()  # Response: Price_US, Predictor: Price_SG
print("\nRegression Model WITHOUT Intercept:")
print(model_without_intercept.summary())

Regression Model WITH Intercept:
                            OLS Regression Results                            
Dep. Variable:               US_Price   R-squared:                       0.898
Model:                            OLS   Adj. R-squared:                  0.873
Method:                 Least Squares   F-statistic:                     35.28
Date:                Tue, 31 Mar 2026   Prob (F-statistic):            0.00403
Time:                        22:00:18   Log-Likelihood:                0.82052
No. Observations:                   6   AIC:                             2.359
Df Residuals:                       4   BIC:                             1.942
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.23

C:\Users\Spike\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
C:\Users\Spike\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 6 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "


## CALCULATING SCALING FACTOR FOR NATURAL GAS

In [98]:
US_ng_data.head()
# SG_ng_data.head()

,Date,Price
0,1997-01-01,3.45
1,1997-02-01,2.15
2,1997-03-01,1.89
3,1997-04-01,2.03
4,1997-05-01,2.25


In [22]:
# Ensure 'date' is in datetime format (if not already)
US_ng_data['Date'] = pd.to_datetime(US_ng_data['Date'])

# Extract the year from the date
US_ng_data['year'] = US_ng_data['Date'].dt.year

# Group by year and calculate the average price
avg_price_per_year_us = US_ng_data.groupby('year')['Price'].mean().reset_index()

# Rename columns for clarity (optional)
avg_price_per_year_us.columns = ['Year', 'Price_US']

# Display the new DataFrame
print(avg_price_per_year_us)

    Year  Price_US
0   1997  2.496667
1   1998  2.090833
2   1999  2.270000
3   2000  4.309167
4   2001  3.956667
5   2002  3.366667
6   2003  5.485833
7   2004  5.900000
8   2005  8.811667
9   2006  6.745000
10  2007  6.976667
11  2008  8.861667
12  2009  3.948333
13  2010  4.386667
14  2011  4.000000
15  2012  2.752500
16  2013  3.728333
17  2014  4.391667
18  2015  2.630000
19  2016  2.515000
20  2017  2.985833
21  2018  3.166667
22  2019  2.565833
23  2020  2.033333
24  2021  3.908333
25  2022  6.418333
26  2023  2.535833
27  2024  2.193333
28  2025  3.526667
29  2026  5.670000


In [23]:
# Ensure 'date' is in datetime format (if not already)
SG_ng_data['observation_date'] = pd.to_datetime(SG_ng_data['observation_date'])

# Extract the year from the date
SG_ng_data['year'] = SG_ng_data['observation_date'].dt.year

# Group by year and calculate the average price
avg_price_per_year_sg = SG_ng_data.groupby('year')['Price'].mean().reset_index()

# Rename columns for clarity (optional)
avg_price_per_year_sg.columns = ['Year', 'Price_SG']

# Display the new DataFrame
print(avg_price_per_year_sg)

    Year   Price_SG
0   2016   7.346364
1   2017   7.247445
2   2018   9.795415
3   2019   5.444624
4   2020   4.366424
5   2021  18.600467
6   2022  33.297022
7   2023  13.463505
8   2024  11.708527
9   2025  12.087750
10  2026  10.593500


In [24]:
combined_df = pd.merge(avg_price_per_year_us, avg_price_per_year_sg, on='Year', how='outer')
print(combined_df)

    Year  Price_US   Price_SG
0   1997  2.496667        NaN
1   1998  2.090833        NaN
2   1999  2.270000        NaN
3   2000  4.309167        NaN
4   2001  3.956667        NaN
5   2002  3.366667        NaN
6   2003  5.485833        NaN
7   2004  5.900000        NaN
8   2005  8.811667        NaN
9   2006  6.745000        NaN
10  2007  6.976667        NaN
11  2008  8.861667        NaN
12  2009  3.948333        NaN
13  2010  4.386667        NaN
14  2011  4.000000        NaN
15  2012  2.752500        NaN
16  2013  3.728333        NaN
17  2014  4.391667        NaN
18  2015  2.630000        NaN
19  2016  2.515000   7.346364
20  2017  2.985833   7.247445
21  2018  3.166667   9.795415
22  2019  2.565833   5.444624
23  2020  2.033333   4.366424
24  2021  3.908333  18.600467
25  2022  6.418333  33.297022
26  2023  2.535833  13.463505
27  2024  2.193333  11.708527
28  2025  3.526667  12.087750
29  2026  5.670000  10.593500


In [27]:
# Rename columns for clarity (optional, based on your description)
combined_df.rename(columns={'PriceA': 'Price_US', 'PriceB': 'Price_SG'}, inplace=True)

# Remove all rows with NAs
combined_df.dropna(inplace=True)

# Regression with intercept (includes a constant term)
X_with_intercept = sm.add_constant(combined_df['Price_SG'])  # Predictor with intercept
model_with_intercept = sm.OLS(combined_df['Price_US'], X_with_intercept).fit()  # Response: Price_US
print("Regression Model WITH Intercept:")
print(model_with_intercept.summary())

# Regression without intercept (no constant term)
model_without_intercept = sm.OLS(combined_df['Price_US'], combined_df['Price_SG']).fit()  # Response: Price_US, Predictor: Price_SG
print("\nRegression Model WITHOUT Intercept:")
print(model_without_intercept.summary())

Regression Model WITH Intercept:
                            OLS Regression Results                            
Dep. Variable:               Price_US   R-squared:                       0.559
Model:                            OLS   Adj. R-squared:                  0.510
Method:                 Least Squares   F-statistic:                     11.42
Date:                Tue, 31 Mar 2026   Prob (F-statistic):            0.00813
Time:                        20:02:04   Log-Likelihood:                -14.475
No. Observations:                  11   AIC:                             32.95
Df Residuals:                       9   BIC:                             33.74
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.80

## CALCULATING WATER SCALING FACTOR

In [95]:
# Reading in data
SG_water_data = pd.read_csv("data/sg_water_prices.csv", header= None)
US_water_data = pd.read_excel("data/us_water_prices.xlsx",skiprows=1, header= 0)

# Converting to price per m3
US_water_data["US_Price_per_m3"] = US_water_data["Price ($/kGal)"]*3.78541178/1000

SG_water_data = SG_water_data.T
SG_water_data.columns = SG_water_data.iloc[0]  # Set the first row as column headers
SG_water_data = SG_water_data[1:]
SG_water_data = SG_water_data[['DataSeries',"Industrial Water Price"]]

# Convert the "Industrial Water Price" column to numeric, coercing errors to NaN
SG_water_data.rename(columns={"Industrial Water Price": "SG_Price_per_m3"}, inplace=True)
SG_water_data["SG_Price_per_m3"] = pd.to_numeric(SG_water_data["SG_Price_per_m3"], errors='coerce')
SG_water_data["Year"] = pd.to_numeric(SG_water_data["DataSeries"], errors='coerce')

# Converting to USD per m3
SG_water_data["SG_Price_per_m3"] = SG_water_data["SG_Price_per_m3"]/1.28


print(SG_water_data.head())

0 DataSeries  SG_Price_per_m3  Year
1       2025         1.367188  2025
2       2024         1.296875  2024
3       2023         1.234375  2023
4       2022         1.234375  2022
5       2021         1.234375  2021


In [99]:
combined_df = pd.merge(SG_water_data, US_water_data, on="Year", how='outer')  # 'outer' includes all rows from both; use 'inner' for matching rows only
combined_df = combined_df[["Year", "SG_Price_per_m3", "US_Price_per_m3"]].dropna()
print(combined_df.head(30))

    Year  SG_Price_per_m3  US_Price_per_m3
12  2008         0.312500         0.010637
14  2010         0.359375         0.011773
16  2012         0.453125         0.012605
18  2014         0.945312         0.013590
20  2016         0.945312         0.014650
23  2019         1.234375         0.015974
25  2021         1.234375         0.014612


In [101]:

# Regression with intercept (includes a constant term)
X_with_intercept = sm.add_constant(combined_df['SG_Price_per_m3'])  # Predictor with intercept
model_with_intercept = sm.OLS(combined_df['US_Price_per_m3'], X_with_intercept).fit()  # Response: US_Price_per_m3
print("Regression Model WITH Intercept:")
print(model_with_intercept.summary())

# Regression without intercept (no constant term)
model_without_intercept = sm.OLS(combined_df['US_Price_per_m3'], combined_df['SG_Price_per_m3']).fit()  # Response: US_Price_per_m3, Predictor: SG_Price_per_m3
print("\nRegression Model WITHOUT Intercept:")
print(model_without_intercept.summary())

Regression Model WITH Intercept:
                            OLS Regression Results                            
Dep. Variable:        US_Price_per_m3   R-squared:                       0.880
Model:                            OLS   Adj. R-squared:                  0.856
Method:                 Least Squares   F-statistic:                     36.72
Date:                Tue, 31 Mar 2026   Prob (F-statistic):            0.00177
Time:                        21:05:26   Log-Likelihood:                 42.068
No. Observations:                   7   AIC:                            -80.14
Df Residuals:                       5   BIC:                            -80.25
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const    

C:\Users\Spike\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 7 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
C:\Users\Spike\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 7 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
